In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

spark = (SparkSession.builder
         .appName("write-udfs")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [0]:
df = (spark.read.format("json")
      .option("multiLine", "true")
      .load("../data/nobel_prizes.json"))

In [0]:
df_flattened = (
    df
    .withColumn("laureates",explode(col("laureates")))
    .select(col("category")
            ,col("year")
            ,col("overallMotivation")
            ,col("laureates.id")
            ,col("laureates.firstname")
            ,col("laureates.surname")
            ,col("laureates.share")
            ,col("laureates.motivation"))
    .filter(col("laureates.firstname").isNotNull() & col("laureates.surname").isNotNull()))

In [0]:
def concat(first_name, last_name):
    return first_name + " " + last_name

In [0]:
from pyspark.sql.functions import udf
concat_udf = udf(concat)

In [0]:
from pyspark.sql.types import StringType
concat_udf = udf(concat, StringType())

In [0]:
df_flattened = df_flattened.withColumn("full_name", concat_udf(df_flattened["firstname"], df_flattened["surname"]))

In [0]:
df_flattened.show()

### Using UDFs in Spark SQL

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Define a UDF
def square_udf(x):
    return x ** 2

# Register the UDF
spark.udf.register("square", square_udf, IntegerType())

# Create a DataFrame
df = spark.createDataFrame([(1,), (2,), (3,), (4,), (5,)], ["num"])

# Use the registered UDF in a SQL query
df.createOrReplaceTempView("numbers")
result = spark.sql("SELECT num, square(num) AS square_num FROM numbers")

# Show the result
result.show()

In [0]:
spark.stop()